# CNN Predict — CESM2-LE JJA SST → September Slowdown

Loads pre-trained CNN models and reproduces the same figures as `cnn_train.ipynb` **without re-training**. Also generates composite maps of the SST input and LRP attributions for TP / FP / TN / FN scenarios.

**Prerequisites**
- `python scripts/04_cesm2le_cnn_train.py` — trains models and saves JSON histories
- `python scripts/05_cesm2le_lrp.py` — computes and saves LRP attributions

**Set `N_SPLITS` and `N_SEEDS` to match the values used during training, then Run All.**

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from configs import paths
from src.cnn.splits import load_tvt_split
from src.cnn.model  import METRIC_NAMES
from src.cnn.train  import load_model, predict_splits

## Config
Must match the values used in `cnn_train.ipynb` (or `scripts/04_cesm2le_cnn_train.py`).

In [ ]:
# ── How many splits / seeds were trained ─────────────────────────────────────
N_SPLITS  = 9          # max 9
N_SEEDS   = 5          # random seeds per split
BASE_SEED = 42

# ── Dataset ───────────────────────────────────────────────────────────────────
BLOCK_SIZE = 10        # ensemble members per TVT block
START_YEAR = 1990      # first year in the loaded splits

METRICS_DIR = paths.RESULTS_DIR / 'metrics'

## Load loop
Loads TVT splits, pre-trained models, metrics, training histories, and LRP attributions.
Results are stored in `results[split_idx]`.

In [ ]:
# results[split_idx] = {
#   'histories':     list[dict]        — one per seed (loaded from JSON)
#   'y_scores_runs': list[dict]        — one per seed, keys: train/val/test
#   'y_true':        dict              — keys: train/val/test
#   'ds_metrics':    xr.Dataset
#   'lrp_data':      list[dict|None]   — one per seed; None if LRP file missing
#   'sst_tr':        np.ndarray        — (n_tr, nx, ny) training SST
#   'lat':           np.ndarray        — (nlat,) 1-D latitude from LRP file
#   'lon':           np.ndarray        — (nlon,) 1-D longitude from LRP file
# }
results = {}

for split_idx in range(N_SPLITS):
    print(f'\n{"─"*60}')
    print(f'Split {split_idx}')
    print(f'{"─"*60}')

    # ── TVT split ─────────────────────────────────────────────────────────────
    split_path = paths.tvt_split_path(split_idx)
    if not split_path.exists():
        raise FileNotFoundError(
            f'TVT split not found: {split_path}\n'
            f'Run scripts/03_cesm2le_tvt_splits.py first.'
        )
    split = load_tvt_split(split_path)

    x_tr = split['sst_tr'][:, :, :, np.newaxis]   # (n_tr, nx, ny, 1)
    x_va = split['sst_va'][:, :, :, np.newaxis]
    x_te = split['sst_te'][:, :, :, np.newaxis]
    y_tr = split['slow_tr']
    y_va = split['slow_va']
    y_te = split['slow_te']
    y_true = {'train': y_tr, 'val': y_va, 'test': y_te}

    print(f'  Train: {x_tr.shape}  prevalence={y_tr.mean():.3f}')
    print(f'  Val  : {x_va.shape}  prevalence={y_va.mean():.3f}')
    print(f'  Test : {x_te.shape}  prevalence={y_te.mean():.3f}')

    histories      = []
    y_scores_runs  = []
    lrp_data       = []
    lat_shared     = None
    lon_shared     = None

    for run_idx in range(N_SEEDS):
        print(f'  seed {BASE_SEED + run_idx}: ', end='', flush=True)

        # ── Model ─────────────────────────────────────────────────────────────
        model_p = paths.model_path(split_idx, run_idx)
        if not model_p.exists():
            raise FileNotFoundError(
                f'Model not found: {model_p}\n'
                f'Run scripts/04_cesm2le_cnn_train.py first.'
            )
        model = load_model(paths.MODELS_DIR, split_idx, run_idx)
        y_scores_runs.append(predict_splits(model, x_tr, x_va, x_te))
        print('model ✓', end='  ', flush=True)

        # ── Training history ──────────────────────────────────────────────────
        hist_path = paths.LOGS_DIR / f'history_split{split_idx}_run{run_idx}.json'
        if hist_path.exists():
            with open(hist_path) as f:
                histories.append(json.load(f))
        else:
            histories.append(None)
            print(f'(history not found: {hist_path})', end='  ')
        print('history ✓', end='  ', flush=True)

        # ── LRP attributions ──────────────────────────────────────────────────
        lrp_path = paths.attribution_path(split_idx, run_idx)
        if lrp_path.exists():
            lrp_ds   = xr.open_dataset(lrp_path)
            lrp_raw  = lrp_ds['lrp_attributions'].values   # (n_chunks, chunk_size, nx, ny, 1)
            lat_1d   = lrp_ds['lat'].values                 # (nx,)
            lon_1d   = lrp_ds['lon'].values                 # (ny,)
            lrp_ds.close()
            # flatten to (n_lrp, nx, ny)
            lrp_flat = lrp_raw.reshape(-1, lrp_raw.shape[2], lrp_raw.shape[3])
            lrp_data.append({'lrp_flat': lrp_flat})
            if lat_shared is None:
                lat_shared = lat_1d
                lon_shared = lon_1d
            print('LRP ✓')
        else:
            lrp_data.append(None)
            print(f'(LRP not found: {lrp_path})')

    # ── Metrics dataset ───────────────────────────────────────────────────────
    metrics_p = paths.metrics_path(split_idx)
    if not metrics_p.exists():
        raise FileNotFoundError(
            f'Metrics not found: {metrics_p}\n'
            f'Run scripts/04_cesm2le_cnn_train.py first.'
        )
    ds_metrics = xr.open_dataset(metrics_p)

    results[split_idx] = {
        'histories':     histories,
        'y_scores_runs': y_scores_runs,
        'y_true':        y_true,
        'ds_metrics':    ds_metrics,
        'lrp_data':      lrp_data,
        'sst_tr':        split['sst_tr'],   # (n_tr, nx, ny) — no channel dim
        'lat':           lat_shared,
        'lon':           lon_shared,
    }

print('\nLoading complete.')

---
## Composite Maps

For each selected scenario (TP / FP / TN / FN):
- **(a)** Mean JJA SST input over all scenario samples (land masked to NaN)
- **(b)** Mean LRP attribution over all scenario samples

Composites are computed per split × seed first, then averaged across all splits × seeds.

> **Note:** LRP must have been computed with `scripts/05_cesm2le_lrp.py`. Splits/seeds missing LRP files are automatically skipped.

In [ ]:
# ── Toggle scenarios to plot ──────────────────────────────────────────────────
SHOW_TP = True
SHOW_FP = False
SHOW_TN = False
SHOW_FN = False

In [ ]:
SCENARIO_FLAGS = {'TP': SHOW_TP, 'FP': SHOW_FP, 'TN': SHOW_TN, 'FN': SHOW_FN}
ACTIVE_SCENARIOS = [k for k, v in SCENARIO_FLAGS.items() if v]

for scenario_name in ACTIVE_SCENARIOS:

    sst_per_sr = []
    lrp_per_sr = []
    lat_plot   = None
    lon_plot   = None

    for split_idx in range(N_SPLITS):
        res    = results[split_idx]
        sst_tr = res['sst_tr']   # (n_tr, nx, ny)

        for run_idx in range(N_SEEDS):
            lrp_info = res['lrp_data'][run_idx]
            if lrp_info is None:
                continue   # LRP not available for this split/seed

            y_s = res['y_scores_runs'][run_idx]['train']
            y_t = res['y_true']['train']

            # ── threshold ────────────────────────────────────────────────────
            prec, rec, thr = precision_recall_curve(y_t, y_s)
            thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
            y_pred   = (y_s >= thr_best).astype(int)

            # ── scenario mask ─────────────────────────────────────────────────
            if scenario_name == 'TP':
                mask = (y_pred == 1) & (y_t == 1)
            elif scenario_name == 'FP':
                mask = (y_pred == 1) & (y_t == 0)
            elif scenario_name == 'TN':
                mask = (y_pred == 0) & (y_t == 0)
            else:   # FN
                mask = (y_pred == 0) & (y_t == 1)

            # ── align LRP with training data (LRP may have fewer samples) ────
            lrp_flat = lrp_info['lrp_flat']   # (n_lrp, nx, ny)
            n_lrp    = lrp_flat.shape[0]
            mask_lrp = mask[:n_lrp]
            sst_lrp  = sst_tr[:n_lrp]         # (n_lrp, nx, ny)

            if mask_lrp.sum() == 0:
                continue

            # ── SST composite — land fill (-10) → NaN ────────────────────────
            sst_masked = sst_lrp.astype(float).copy()
            sst_masked[sst_masked < -5] = np.nan
            sst_mean = np.nanmean(sst_masked[mask_lrp], axis=0)   # (nx, ny)

            # ── LRP composite ────────────────────────────────────────────────
            lrp_mean = np.mean(lrp_flat[mask_lrp], axis=0)        # (nx, ny)

            sst_per_sr.append(sst_mean)
            lrp_per_sr.append(lrp_mean)

            if lat_plot is None:
                lat_plot = res['lat']
                lon_plot = res['lon']

    if len(sst_per_sr) == 0:
        print(f'{scenario_name}: no samples found across any split/seed — skipping.')
        continue

    n_sr      = len(sst_per_sr)
    sst_grand = np.nanmean(np.stack(sst_per_sr, axis=0), axis=0)  # (nx, ny)
    lrp_grand = np.nanmean(np.stack(lrp_per_sr, axis=0), axis=0)  # (nx, ny)

    print(f'{scenario_name}: averaged over {n_sr} split×seed composites')

    # ── Build 2-D lat/lon grid for plotting ───────────────────────────────────
    lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)   # (nlat, nlon)

    # ── Symmetric colour limits at 97th percentile ────────────────────────────
    sst_vmax = np.nanpercentile(np.abs(sst_grand), 97)
    lrp_vmax = np.nanpercentile(np.abs(lrp_grand), 97)

    # ── Figure ────────────────────────────────────────────────────────────────
    proj  = ccrs.PlateCarree(central_longitude=180)

    fig = plt.figure(figsize=(14, 5))
    ax_sst = fig.add_subplot(1, 2, 1, projection=proj)
    ax_lrp = fig.add_subplot(1, 2, 2, projection=proj)

    for ax, data, vmax, cmap, title in [
        (ax_sst, sst_grand, sst_vmax, cmocean.cm.balance, f'(a) Mean SST — {scenario_name}'),
        (ax_lrp, lrp_grand, lrp_vmax, cmocean.cm.curl,           f'(b) Mean LRP — {scenario_name}'),
    ]:
        ax.set_global()
        ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
        ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

        im = ax.pcolormesh(
            lon2d, lat2d, data,
            cmap=cmap,
            vmin=-vmax, vmax=vmax,
            transform=ccrs.PlateCarree(),
            shading='auto',
            zorder=0,
        )
        plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.04,
                     fraction=0.046, label='Standardised SST' if ax is ax_sst else 'Relevance')
        ax.set_title(title, fontsize=11)

    fig.suptitle(
        f'Composites — {scenario_name} training samples\n'
        f'({N_SPLITS} splits × {N_SEEDS} seeds, n_composites={n_sr})',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------------
# plotting options
# ------------------------------------------------------------------
LRP_POSITIVE_ONLY = True   # False = signed relevance, True = keep only positive
LRP_POSITIVE_CMAP = 'magma'
LRP_SIGNED_CMAP   = cmocean.cm.curl

SCENARIO_FLAGS = {'TP': SHOW_TP, 'FP': SHOW_FP, 'TN': SHOW_TN, 'FN': SHOW_FN}
ACTIVE_SCENARIOS = [k for k, v in SCENARIO_FLAGS.items() if v]

for scenario_name in ACTIVE_SCENARIOS:

    sst_per_sr = []
    lrp_per_sr = []
    lat_plot   = None
    lon_plot   = None

    for split_idx in range(N_SPLITS):
        res    = results[split_idx]
        sst_tr = res['sst_tr']   # (n_tr, nx, ny)

        for run_idx in range(N_SEEDS):
            lrp_info = res['lrp_data'][run_idx]
            if lrp_info is None:
                continue

            y_s = res['y_scores_runs'][run_idx]['train']
            y_t = res['y_true']['train']

            # ── threshold ──────────────────────────────────────────────
            prec, rec, thr = precision_recall_curve(y_t, y_s)
            thr_best = thr[np.argmin(np.abs(prec[:-1] - rec[:-1]))] if thr.size > 0 else 0.5
            y_pred   = (y_s >= thr_best).astype(int)

            # ── scenario mask ──────────────────────────────────────────
            if scenario_name == 'TP':
                mask = (y_pred == 1) & (y_t == 1)
            elif scenario_name == 'FP':
                mask = (y_pred == 1) & (y_t == 0)
            elif scenario_name == 'TN':
                mask = (y_pred == 0) & (y_t == 0)
            else:   # FN
                mask = (y_pred == 0) & (y_t == 1)

            # ── align LRP with training data ───────────────────────────
            lrp_flat = lrp_info['lrp_flat']   # (n_lrp, nx, ny)
            n_lrp    = lrp_flat.shape[0]
            mask_lrp = mask[:n_lrp]
            sst_lrp  = sst_tr[:n_lrp]

            if mask_lrp.sum() == 0:
                continue

            # ── SST composite — land fill (-10) → NaN ─────────────────
            sst_masked = sst_lrp.astype(float).copy()
            sst_masked[sst_masked < -5] = np.nan
            sst_mean = np.nanmean(sst_masked[mask_lrp], axis=0)

            # ── LRP composite ──────────────────────────────────────────
            lrp_selected = lrp_flat[mask_lrp]

            if LRP_POSITIVE_ONLY:
                lrp_selected = np.where(lrp_selected > 0, lrp_selected, 0.0)

            lrp_mean = np.mean(lrp_selected, axis=0)

            sst_per_sr.append(sst_mean)
            lrp_per_sr.append(lrp_mean)

            if lat_plot is None:
                lat_plot = res['lat']
                lon_plot = res['lon']

    if len(sst_per_sr) == 0:
        print(f'{scenario_name}: no samples found across any split/seed — skipping.')
        continue

    n_sr      = len(sst_per_sr)
    sst_grand = np.nanmean(np.stack(sst_per_sr, axis=0), axis=0)
    lrp_grand = np.nanmean(np.stack(lrp_per_sr, axis=0), axis=0)

    print(f'{scenario_name}: averaged over {n_sr} split×seed composites')

    # ── Build 2-D lat/lon grid for plotting ───────────────────────────
    lon2d, lat2d = np.meshgrid(lon_plot, lat_plot)

    # ── Colour limits ─────────────────────────────────────────────────
    sst_vmax = np.nanpercentile(np.abs(sst_grand), 97)

    if LRP_POSITIVE_ONLY:
        lrp_vmin = 0.0
        lrp_vmax = np.nanpercentile(lrp_grand, 97)
        lrp_cmap = LRP_POSITIVE_CMAP
    else:
        lrp_vmax = np.nanpercentile(np.abs(lrp_grand), 97)
        lrp_vmin = -lrp_vmax
        lrp_cmap = LRP_SIGNED_CMAP

    # ── Figure ────────────────────────────────────────────────────────
    proj = ccrs.PlateCarree(central_longitude=180)

    fig = plt.figure(figsize=(14, 5))
    ax_sst = fig.add_subplot(1, 2, 1, projection=proj)
    ax_lrp = fig.add_subplot(1, 2, 2, projection=proj)

    # SST
    ax_sst.set_global()
    ax_sst.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_sst.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    ax_sst.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im_sst = ax_sst.pcolormesh(
        lon2d, lat2d, sst_grand,
        cmap=cmocean.cm.balance,
        vmin=-sst_vmax, vmax=sst_vmax,
        transform=ccrs.PlateCarree(),
        shading='auto',
        zorder=0,
    )
    plt.colorbar(
        im_sst, ax=ax_sst, orientation='horizontal', pad=0.04,
        fraction=0.046, label='Standardised SST'
    )
    ax_sst.set_title(f'(a) Mean SST — {scenario_name}', fontsize=11)

    # LRP
    ax_lrp.set_global()
    ax_lrp.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
    ax_lrp.add_feature(cfeature.COASTLINE, linewidth=0.4, zorder=2)
    ax_lrp.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

    im_lrp = ax_lrp.pcolormesh(
        lon2d, lat2d, lrp_grand,
        cmap=lrp_cmap,
        vmin=lrp_vmin, vmax=lrp_vmax,
        transform=ccrs.PlateCarree(),
        shading='auto',
        zorder=0,
    )
    plt.colorbar(
        im_lrp, ax=ax_lrp, orientation='horizontal', pad=0.04,
        fraction=0.046,
        label='Positive relevance' if LRP_POSITIVE_ONLY else 'Relevance'
    )
    ax_lrp.set_title(
        f"(b) Mean LRP — {scenario_name}" + (" (positive only)" if LRP_POSITIVE_ONLY else ""),
        fontsize=11
    )

    fig.suptitle(
        f'Composites — {scenario_name} training samples\n'
        f'({N_SPLITS} splits × {N_SEEDS} seeds, n_composites={n_sr})',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.show()

---
## ENSO-Indexed Composite Maps

For each selected scenario (TP / FP / TN / FN):
- **top row (a-c)** Mean JJA SST input over -/0/+ Nino Index scenario samples (land masked to NaN)
- **bottom row (d-f)** Mean LRP attribution over -/0/+ Nino Index scenario samples

Composites are computed per split × seed first, then averaged across all splits × seeds.

> **Note:** LRP must have been computed with `scripts/05_cesm2le_lrp.py`. Splits/seeds missing LRP files are automatically skipped.